# Lab 2 Phase 5A: Wide SE Super-Resolution (TPU-Compatible Colab Recipe)

Self-contained Colab notebook for a TPU-compatible version of the Phase 5A Wide SE model. Training and evaluation run on TPU via PyTorch/XLA. ONNX export, parity checks, diagnostics, and calibration export run on CPU after training. MXQ compilation remains a separate post-export step.

This notebook is fully self-contained for Colab. It does not import sibling repo files.
Output directory: `Lab 2 Phase 5/runs/phase5a_wide_se_tpu`


## 1. Environment And Package Setup


In [ ]:

import importlib.util
import os
import subprocess
import sys

REQUIRED_PACKAGES = {
    "onnx": "onnx",
    "onnxruntime": "onnxruntime",
}
missing = [pkg for module_name, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required CPU export packages already available.")

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB and importlib.util.find_spec("torch_xla") is None:
    print("Installing torch_xla[tpu] for Colab TPU runtime...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch_xla[tpu]",
        "-f",
        "https://storage.googleapis.com/libtpu-releases/index.html",
    ])
    print("Installed torch_xla[tpu]. If Colab still cannot import torch_xla, restart the runtime once.")
elif IN_COLAB:
    print("torch_xla already available.")
else:
    print("Skipping torch_xla install outside Colab.")


In [ ]:

from __future__ import annotations

import hashlib
import io
import json
import math
import os
import random
import shutil
import subprocess
import tarfile
import textwrap
import time
import warnings
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Any

warnings.filterwarnings("ignore", message=".*legacy TorchScript-based ONNX.*")

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFilter, ImageOps
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from torchvision import transforms

try:
    import onnx
except ImportError:
    onnx = None

try:
    import onnxruntime as ort
except ImportError:
    ort = None

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


## 2. Drive Mount And Workspace Staging


In [ ]:

OUTPUT_SUBDIR = "phase5a_wide_se_tpu"
DRIVE_ROOT = "/content/drive/MyDrive/Data 255 Class Spring 2026/Lab 2"
DRIVE_ROOT_CANDIDATES = [
    DRIVE_ROOT,
    "/content/drive/MyDrive/Data 255 Spring 2026/Lab 2",
    "/content/drive/MyDrive/Lab-2-colab",
    "/content/drive/MyDrive/Colab Notebooks/Lab-2-colab",
]

EPOCHS = 80
GLOBAL_BATCH_SIZE = 128
HOST_DATALOADER_WORKERS = 8
IMAGENET_TRAIN_LIMIT = 6000
IMAGENET_VAL_LIMIT = 300
TRAIN_EVAL_SUBSET_SIZE = 128
RESUME_TRAINING = True
SEED = 255
CHECKPOINT_INTERVAL = 10
TRAIN_EVAL_INTERVAL = 5
EARLY_STOP_PATIENCE = 15
XLA_DEBUG_SINGLE_PROCESS = False
CPU_EXPORT_BATCH_SIZE = 8

NOTEBOOK_DIR = Path(os.path.abspath("")).resolve()
DRIVE_DATA_ROOT = None
COLAB_SYNC_ROOT = None


def first_existing(*paths: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return paths[0]


def ensure_tar_extracted(archive_path: Path, dest_root: Path) -> None:
    if not archive_path.exists():
        return
    print(f"Extracting {archive_path.name} -> {dest_root}")
    dest_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:*") as tf:
        tf.extractall(dest_root)


def ensure_unzipped(zip_path: Path, extracted_dir: Path) -> Path:
    if extracted_dir.exists():
        return extracted_dir
    if not zip_path.exists():
        return extracted_dir
    print(f"Extracting {zip_path.name} -> {zip_path.parent}")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(zip_path.parent)
    return extracted_dir


if IN_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive", force_remount=False)
    mydrive_root = Path("/content/drive/MyDrive")
    course_lab_root = first_existing(
        mydrive_root / "Data 255 Class Spring 2026" / "Lab 2",
        mydrive_root / "Data 255 Spring 2026" / "Lab 2",
    )
    fallback_sync_root = first_existing(
        mydrive_root / "Lab-2-colab",
        mydrive_root / "Colab Notebooks" / "Lab-2-colab",
    )
    if course_lab_root.exists():
        COLAB_SYNC_ROOT = course_lab_root
    else:
        COLAB_SYNC_ROOT = fallback_sync_root
        COLAB_SYNC_ROOT.mkdir(parents=True, exist_ok=True)

    PROJECT_ROOT = Path(f"/content/{OUTPUT_SUBDIR}_workspace")
    data_archive = first_existing(
        COLAB_SYNC_ROOT / f"{OUTPUT_SUBDIR}_colab_data.tar",
        COLAB_SYNC_ROOT / f"{OUTPUT_SUBDIR}_colab_data.tar.gz",
        COLAB_SYNC_ROOT / "lab2_phase5_data.tar",
        COLAB_SYNC_ROOT / "lab2_phase5_data.tar.gz",
        COLAB_SYNC_ROOT / "lab2_colab_data.tar",
        COLAB_SYNC_ROOT / "lab2_colab_data.tar.gz",
    )
    DRIVE_DATA_ROOT = first_existing(
        COLAB_SYNC_ROOT / "Data",
        course_lab_root / "Data",
    )
    local_data_root = PROJECT_ROOT / "Data"

    if not local_data_root.exists():
        if data_archive.exists():
            ensure_tar_extracted(data_archive, PROJECT_ROOT)
        elif DRIVE_DATA_ROOT.exists():
            print(f"Using Drive-backed Data directory: {DRIVE_DATA_ROOT}")
        else:
            raise FileNotFoundError(
                f"Could not find a Data directory or staged archive under {COLAB_SYNC_ROOT}."
            )

    OUTPUT_DIR = COLAB_SYNC_ROOT / "runs" / OUTPUT_SUBDIR
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    PROJECT_ROOT = NOTEBOOK_DIR
    probe = NOTEBOOK_DIR
    while probe != probe.parent and not (probe / "Data").exists():
        probe = probe.parent
    if not (probe / "Data").exists():
        raise FileNotFoundError("Could not locate repository Data directory from notebook location.")
    PROJECT_ROOT = probe
    OUTPUT_DIR = PROJECT_ROOT / "Lab 2 Phase 5" / "runs" / OUTPUT_SUBDIR
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_DATA_ROOT = PROJECT_ROOT / "Data"

DATA_ROOT = (PROJECT_ROOT / "Data") if (PROJECT_ROOT / "Data").exists() else DRIVE_DATA_ROOT
if IN_COLAB and DATA_ROOT.exists() and DRIVE_DATA_ROOT and DRIVE_DATA_ROOT.exists():
    if not ((DATA_ROOT / "HR_train").exists() and (DATA_ROOT / "LR_train").exists()):
        print(f"Falling back to Drive-backed Data directory because staged data is incomplete: {DATA_ROOT}")
        DATA_ROOT = DRIVE_DATA_ROOT

COURSE_EXPORT_ROOT = DATA_ROOT / "course_files_export"
LEGACY_IMAGE_ROOT = DATA_ROOT / "ImageNet"
HR_TRAIN_ROOT = DATA_ROOT / "HR_train"
LR_TRAIN_ROOT = DATA_ROOT / "LR_train"
HR_VAL_DIR = first_existing(DATA_ROOT / "HR_val", DATA_ROOT / "val" / "HR_val")
LR_VAL_DIR = first_existing(DATA_ROOT / "LR_val", DATA_ROOT / "val" / "LR_val")
IMAGENET_TRAIN_LIST = first_existing(COURSE_EXPORT_ROOT / "imagenet_train20.txt", LEGACY_IMAGE_ROOT / "imagenet_train20.txt")
IMAGENET_VAL_LIST = first_existing(COURSE_EXPORT_ROOT / "imagenet_val20.txt", LEGACY_IMAGE_ROOT / "imagenet_val20.txt")
IMAGENET_TRAIN_ROOT = ensure_unzipped(
    COURSE_EXPORT_ROOT / "imagenet_train20.zip",
    first_existing(COURSE_EXPORT_ROOT / "imagenet_train20a", LEGACY_IMAGE_ROOT / "imagenet_train20a"),
)
IMAGENET_VAL_ROOT = ensure_unzipped(
    COURSE_EXPORT_ROOT / "imagenet_val20.zip",
    first_existing(COURSE_EXPORT_ROOT / "imagenet_val20", LEGACY_IMAGE_ROOT / "imagenet_val20"),
)

required_paths = [
    DATA_ROOT,
    HR_TRAIN_ROOT,
    LR_TRAIN_ROOT,
    HR_VAL_DIR,
    LR_VAL_DIR,
    IMAGENET_TRAIN_LIST,
    IMAGENET_VAL_LIST,
]
missing = [path for path in required_paths if not path.exists()]
if missing:
    joined = ", ".join(str(path) for path in missing)
    raise FileNotFoundError(f"Missing required Lab 2 data paths: {joined}")

workspace = {
    "repo_root": PROJECT_ROOT,
    "workspace_root": PROJECT_ROOT,
    "data_root": DATA_ROOT,
    "output_dir": OUTPUT_DIR,
    "in_colab": IN_COLAB,
}


## 3. Data Discovery And Dataset Config


In [ ]:

paths_cfg = {
    "data_root": str(DATA_ROOT),
    "hr_train_root": str(HR_TRAIN_ROOT),
    "lr_train_root": str(LR_TRAIN_ROOT),
    "hr_val_dir": str(HR_VAL_DIR),
    "lr_val_dir": str(LR_VAL_DIR),
    "imagenet_train_root": str(IMAGENET_TRAIN_ROOT),
    "imagenet_val_root": str(IMAGENET_VAL_ROOT),
    "imagenet_train_list": str(IMAGENET_TRAIN_LIST),
    "imagenet_val_list": str(IMAGENET_VAL_LIST),
}

train_lr_count = len(list(LR_TRAIN_ROOT.glob("*/*.png")))
train_hr_count = len(list(HR_TRAIN_ROOT.glob("*/*.png")))
val_lr_count = len(list(LR_VAL_DIR.glob("*.png")))
val_hr_count = len(list(HR_VAL_DIR.glob("*.png")))
imagenet_train_manifest_count = len([line for line in IMAGENET_TRAIN_LIST.read_text().splitlines() if line.strip()])
imagenet_val_manifest_count = len([line for line in IMAGENET_VAL_LIST.read_text().splitlines() if line.strip()])

print(json.dumps({
    "paired_train_lr": train_lr_count,
    "paired_train_hr": train_hr_count,
    "paired_val_lr": val_lr_count,
    "paired_val_hr": val_hr_count,
    "hr_train_root": str(HR_TRAIN_ROOT),
    "lr_train_root": str(LR_TRAIN_ROOT),
    "hr_val_dir": str(HR_VAL_DIR),
    "lr_val_dir": str(LR_VAL_DIR),
    "imagenet_train_root": str(IMAGENET_TRAIN_ROOT),
    "imagenet_val_root": str(IMAGENET_VAL_ROOT),
    "imagenet_train_manifest": str(IMAGENET_TRAIN_LIST),
    "imagenet_train_manifest_count": imagenet_train_manifest_count,
    "imagenet_val_manifest": str(IMAGENET_VAL_LIST),
    "imagenet_val_manifest_count": imagenet_val_manifest_count,
}, indent=2))


## 4. Shared Runtime/Training Helpers


In [ ]:

COMMON_SOURCE = textwrap.dedent(r"""
TO_TENSOR = transforms.ToTensor()
BICUBIC = Image.Resampling.BICUBIC if hasattr(Image, 'Resampling') else Image.BICUBIC
FORBIDDEN_TYPES = (nn.ReLU, nn.Sigmoid, nn.Softmax, nn.LayerNorm, nn.GroupNorm)


def default_data_cfg() -> dict[str, Any]:
    return {
        'train_patch_size': 224,
        'eval_size': 256,
        'random_scale_pad': 32,
        'cutout_prob': 0.35,
        'cutout_ratio': 0.18,
        'lr_noise_prob': 0.30,
        'lr_noise_std': 0.015,
        'lr_blur_prob': 0.70,
        'jpeg_prob': 0.50,
        'jpeg_quality_min': 40,
        'jpeg_quality_max': 90,
        'downsample_scales': (2, 3, 4),
        'imagenet_train_limit': 6000,
        'imagenet_val_limit': 300,
        'train_eval_subset_size': 128,
    }


def default_train_cfg(global_batch_size: int) -> dict[str, Any]:
    return {
        'epochs': 80,
        'global_batch_size': global_batch_size,
        'lr': 3e-4,
        'weight_decay': 2e-4,
        'warmup_epochs': 5,
        'min_lr_ratio': 0.05,
        'checkpoint_interval': 10,
        'train_eval_interval': 5,
        'seed': 255,
        'early_stop_patience': 15,
        'grad_clip_norm': 1.0,
        'ema_decay': 0.999,
        'charb_eps': 1e-6,
        'use_amp': True,
    }


def validate_paths(paths_cfg: dict[str, Any]) -> dict[str, Path]:
    resolved = {key: Path(value) for key, value in paths_cfg.items()}
    required = [
        'data_root',
        'hr_train_root',
        'lr_train_root',
        'hr_val_dir',
        'lr_val_dir',
        'imagenet_train_root',
        'imagenet_val_root',
        'imagenet_train_list',
        'imagenet_val_list',
    ]
    missing = [str(resolved[key]) for key in required if not resolved[key].exists()]
    if missing:
        raise FileNotFoundError(f"Missing required data paths: {', '.join(missing)}")
    return resolved


def collect_paired_by_subfolder(lr_root: Path, hr_root: Path) -> list[tuple[Path, Path, str]]:
    pairs: list[tuple[Path, Path, str]] = []
    for hr_dir in sorted(p for p in hr_root.iterdir() if p.is_dir()):
        suffix = hr_dir.name.replace('HR_train', '')
        lr_dir = lr_root / f'LR_train{suffix}'
        if not lr_dir.exists():
            continue
        hr_imgs = {p.stem: p for p in sorted(hr_dir.glob('*.png'))}
        lr_imgs = {p.stem: p for p in sorted(lr_dir.glob('*.png'))}
        common = sorted(set(hr_imgs) & set(lr_imgs))
        pairs.extend((lr_imgs[s], hr_imgs[s], f'{hr_dir.name}/{s}') for s in common)
    return pairs


def collect_paired_flat(lr_dir: Path, hr_dir: Path) -> list[tuple[Path, Path, str]]:
    hr_imgs = {p.stem: p for p in sorted(hr_dir.glob('*.png'))}
    lr_imgs = {p.stem: p for p in sorted(lr_dir.glob('*.png'))}
    common = sorted(set(hr_imgs) & set(lr_imgs))
    return [(lr_imgs[s], hr_imgs[s], s) for s in common]


def read_imagenet_manifest(path: Path) -> list[tuple[str, int]]:
    rows = []
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 2:
            continue
        rows.append((parts[0], int(parts[1])))
    return rows


def collect_imagenet_records(rows: list[tuple[str, int]], root: Path, split: str) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    missing = 0
    for filename, class_id in rows:
        synset = filename.split('_')[0]
        path = (root / synset / filename) if split == 'train' else (root / filename)
        if not path.exists():
            missing += 1
            continue
        records.append(
            {
                'path': path,
                'stem': path.stem,
                'class_id': class_id,
                'split': split,
                'synset': synset,
            }
        )
    if missing:
        print(f"WARNING: skipped {missing} missing ImageNet {split} files")
    return records


def seeded_rng(key: str) -> random.Random:
    digest = hashlib.sha256(key.encode('utf-8')).hexdigest()
    return random.Random(int(digest[:16], 16))


def take_manifest_subset(records: list[Any], limit: int | None, seed: int) -> list[Any]:
    if limit is None or limit >= len(records):
        return list(records)
    rng = random.Random(seed)
    items = list(records)
    rng.shuffle(items)
    return items[:limit]


def random_crop_pair(lr_img: Image.Image, hr_img: Image.Image, size: int, rng: random.Random):
    if lr_img.width == size and lr_img.height == size:
        return lr_img, hr_img
    x0 = rng.randint(0, lr_img.width - size)
    y0 = rng.randint(0, lr_img.height - size)
    box = (x0, y0, x0 + size, y0 + size)
    return lr_img.crop(box), hr_img.crop(box)


def random_crop_single(img: Image.Image, size: int, rng: random.Random):
    if img.width == size and img.height == size:
        return img
    x0 = rng.randint(0, img.width - size)
    y0 = rng.randint(0, img.height - size)
    return img.crop((x0, y0, x0 + size, y0 + size))


def augment_pair(lr_img: Image.Image, hr_img: Image.Image, rng: random.Random):
    if rng.random() > 0.5:
        lr_img = lr_img.transpose(Image.FLIP_LEFT_RIGHT)
        hr_img = hr_img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() > 0.5:
        lr_img = lr_img.transpose(Image.FLIP_TOP_BOTTOM)
        hr_img = hr_img.transpose(Image.FLIP_TOP_BOTTOM)
    k = rng.randint(0, 3)
    if k > 0:
        rot = {1: Image.ROTATE_90, 2: Image.ROTATE_180, 3: Image.ROTATE_270}[k]
        lr_img = lr_img.transpose(rot)
        hr_img = hr_img.transpose(rot)
    return lr_img, hr_img


def augment_single(img: Image.Image, rng: random.Random):
    if rng.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() > 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    k = rng.randint(0, 3)
    if k > 0:
        img = img.transpose({1: Image.ROTATE_90, 2: Image.ROTATE_180, 3: Image.ROTATE_270}[k])
    return img


def jpeg_roundtrip(img: Image.Image, quality: int) -> Image.Image:
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')


def degrade_from_hr(hr_img: Image.Image, rng: random.Random, cfg: dict[str, Any]) -> Image.Image:
    lr_img = hr_img.copy()
    if rng.random() < cfg['lr_blur_prob']:
        lr_img = lr_img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.2, 1.2)))
    scale = rng.choice(cfg['downsample_scales'])
    small = (max(1, hr_img.width // scale), max(1, hr_img.height // scale))
    lr_img = lr_img.resize(small, resample=BICUBIC).resize(hr_img.size, resample=BICUBIC)
    if rng.random() < cfg['jpeg_prob']:
        lr_img = jpeg_roundtrip(lr_img, rng.randint(cfg['jpeg_quality_min'], cfg['jpeg_quality_max']))
    return lr_img


def apply_tensor_regularization(lr_t: torch.Tensor, rng: random.Random, cfg: dict[str, Any], train: bool) -> torch.Tensor:
    if not train:
        return lr_t
    if rng.random() < cfg['lr_noise_prob']:
        lr_t = (lr_t + torch.randn_like(lr_t) * cfg['lr_noise_std']).clamp(0.0, 1.0)
    if rng.random() < cfg['cutout_prob']:
        _, h, w = lr_t.shape
        cut = max(8, int(min(h, w) * cfg['cutout_ratio']))
        x0 = rng.randint(0, w - cut)
        y0 = rng.randint(0, h - cut)
        fill = lr_t.mean().item()
        lr_t[:, y0 : y0 + cut, x0 : x0 + cut] = fill
    return lr_t


class PairedSRDataset(Dataset):
    def __init__(self, pairs, train: bool, data_cfg: dict[str, Any], source_name: str, seed: int):
        self.pairs = pairs
        self.train = train
        self.data_cfg = data_cfg
        self.source_name = source_name
        self.seed = seed

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        lr_path, hr_path, stem = self.pairs[idx]
        lr_img = Image.open(lr_path).convert('RGB')
        hr_img = Image.open(hr_path).convert('RGB')
        rng = random.Random(self.seed + idx) if self.train else seeded_rng(f'{self.source_name}:{stem}')
        if self.train:
            lr_img, hr_img = random_crop_pair(lr_img, hr_img, self.data_cfg['train_patch_size'], rng)
            lr_img, hr_img = augment_pair(lr_img, hr_img, rng)
        else:
            lr_img = ImageOps.fit(lr_img, (self.data_cfg['eval_size'], self.data_cfg['eval_size']), method=BICUBIC)
            hr_img = ImageOps.fit(hr_img, (self.data_cfg['eval_size'], self.data_cfg['eval_size']), method=BICUBIC)
        lr_t = TO_TENSOR(lr_img)
        hr_t = TO_TENSOR(hr_img)
        lr_t = apply_tensor_regularization(lr_t, rng, self.data_cfg, train=self.train)
        return lr_t, hr_t, stem, self.source_name


class ImageNetSyntheticSRDataset(Dataset):
    def __init__(self, records, train: bool, data_cfg: dict[str, Any], source_name: str, seed: int):
        self.records = records
        self.train = train
        self.data_cfg = data_cfg
        self.source_name = source_name
        self.seed = seed

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        hr_img = Image.open(record['path']).convert('RGB')
        rng = random.Random(self.seed + idx) if self.train else seeded_rng(f"{self.source_name}:{record['stem']}")
        if self.train:
            base_size = max(self.data_cfg['eval_size'], self.data_cfg['train_patch_size'] + self.data_cfg['random_scale_pad'])
            hr_img = ImageOps.fit(hr_img, (base_size, base_size), method=BICUBIC)
            hr_img = random_crop_single(hr_img, self.data_cfg['train_patch_size'], rng)
            hr_img = augment_single(hr_img, rng)
        else:
            hr_img = ImageOps.fit(hr_img, (self.data_cfg['eval_size'], self.data_cfg['eval_size']), method=BICUBIC)
        lr_img = degrade_from_hr(hr_img, rng, self.data_cfg)
        lr_t = TO_TENSOR(lr_img)
        hr_t = TO_TENSOR(hr_img)
        lr_t = apply_tensor_regularization(lr_t, rng, self.data_cfg, train=self.train)
        return lr_t, hr_t, record['stem'], self.source_name


def build_datasets(paths_cfg: dict[str, Any], data_cfg: dict[str, Any], seed: int) -> dict[str, Any]:
    paths = validate_paths(paths_cfg)
    train_pairs = collect_paired_by_subfolder(paths['lr_train_root'], paths['hr_train_root'])
    val_pairs = collect_paired_flat(paths['lr_val_dir'], paths['hr_val_dir'])
    if not train_pairs:
        raise FileNotFoundError(f"No paired training PNGs found under {paths['hr_train_root']} and {paths['lr_train_root']}")
    if not val_pairs:
        raise FileNotFoundError(f"No paired validation PNGs found under {paths['hr_val_dir']} and {paths['lr_val_dir']}")

    imagenet_train_rows = read_imagenet_manifest(paths['imagenet_train_list'])
    imagenet_val_rows = read_imagenet_manifest(paths['imagenet_val_list'])
    imagenet_train_records = collect_imagenet_records(imagenet_train_rows, paths['imagenet_train_root'], split='train')
    imagenet_val_records = collect_imagenet_records(imagenet_val_rows, paths['imagenet_val_root'], split='val')
    imagenet_train_used = take_manifest_subset(imagenet_train_records, data_cfg['imagenet_train_limit'], seed)
    imagenet_val_used = take_manifest_subset(imagenet_val_records, data_cfg['imagenet_val_limit'], seed)

    paired_train_dataset = PairedSRDataset(train_pairs, train=True, data_cfg=data_cfg, source_name='paired_train', seed=seed)
    paired_train_eval_dataset = PairedSRDataset(train_pairs, train=False, data_cfg=data_cfg, source_name='paired_train', seed=seed)
    paired_val_dataset = PairedSRDataset(val_pairs, train=False, data_cfg=data_cfg, source_name='paired_val', seed=seed)
    imagenet_train_dataset = ImageNetSyntheticSRDataset(imagenet_train_used, train=True, data_cfg=data_cfg, source_name='imagenet_train', seed=seed)
    imagenet_train_eval_dataset = ImageNetSyntheticSRDataset(imagenet_train_used, train=False, data_cfg=data_cfg, source_name='imagenet_train', seed=seed)
    imagenet_val_dataset = ImageNetSyntheticSRDataset(imagenet_val_used, train=False, data_cfg=data_cfg, source_name='imagenet_val', seed=seed)

    train_dataset = ConcatDataset([paired_train_dataset, imagenet_train_dataset])
    train_eval_dataset = ConcatDataset([paired_train_eval_dataset, imagenet_train_eval_dataset])
    val_dataset = ConcatDataset([paired_val_dataset, imagenet_val_dataset])

    return {
        'train_pairs': train_pairs,
        'val_pairs': val_pairs,
        'imagenet_train_records': imagenet_train_records,
        'imagenet_val_records': imagenet_val_records,
        'train_dataset': train_dataset,
        'train_eval_dataset': train_eval_dataset,
        'val_dataset': val_dataset,
        'paired_val_dataset': paired_val_dataset,
        'imagenet_val_dataset': imagenet_val_dataset,
        'calibration_datasets': {
            'paired_train': paired_train_eval_dataset,
            'imagenet_train': imagenet_train_eval_dataset,
        },
    }


def loader_kwargs(num_workers: int, pin_memory: bool, prefetch_factor: int = 4) -> dict[str, Any]:
    kwargs: dict[str, Any] = {'num_workers': num_workers, 'pin_memory': pin_memory}
    if num_workers > 0:
        kwargs['persistent_workers'] = True
        kwargs['prefetch_factor'] = prefetch_factor
    return kwargs


def make_fixed_subset_loader(dataset, subset_size: int, batch_size: int, seed: int, num_workers: int, pin_memory: bool):
    subset_size = min(subset_size, len(dataset))
    rng = random.Random(seed)
    indices = list(range(len(dataset)))
    rng.shuffle(indices)
    subset = Subset(dataset, indices[:subset_size])
    return DataLoader(subset, batch_size=batch_size, shuffle=False, **loader_kwargs(num_workers, pin_memory))


def build_data_bundle(paths_cfg: dict[str, Any], data_cfg: dict[str, Any], batch_size: int, num_workers: int, seed: int, pin_memory: bool = False) -> dict[str, Any]:
    bundle = build_datasets(paths_cfg, data_cfg, seed)
    train_loader = DataLoader(
        bundle['train_dataset'],
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        **loader_kwargs(num_workers, pin_memory),
    )
    train_eval_loader = make_fixed_subset_loader(
        bundle['train_eval_dataset'],
        data_cfg['train_eval_subset_size'],
        batch_size,
        seed,
        num_workers,
        pin_memory,
    )
    val_loader = DataLoader(bundle['val_dataset'], batch_size=batch_size, shuffle=False, **loader_kwargs(num_workers, pin_memory))
    paired_val_loader = DataLoader(bundle['paired_val_dataset'], batch_size=batch_size, shuffle=False, **loader_kwargs(num_workers, pin_memory))
    imagenet_val_loader = DataLoader(bundle['imagenet_val_dataset'], batch_size=batch_size, shuffle=False, **loader_kwargs(num_workers, pin_memory))
    bundle.update(
        {
            'train_loader': train_loader,
            'train_eval_loader': train_eval_loader,
            'val_loader': val_loader,
            'paired_val_loader': paired_val_loader,
            'imagenet_val_loader': imagenet_val_loader,
        }
    )
    return bundle


def print_data_summary(bundle: dict[str, Any]) -> None:
    lr_batch, hr_batch, _, _ = next(iter(bundle['train_loader']))
    print(f"Combined train: {len(bundle['train_dataset'])}")
    print(f"Combined val:   {len(bundle['val_dataset'])}")
    print(f"Train-eval subset: {len(bundle['train_eval_loader'].dataset)}")
    print(f"Batch shapes: LR={tuple(lr_batch.shape)}, HR={tuple(hr_batch.shape)}")


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def assert_npu_compatible(model: nn.Module) -> None:
    for name, mod in model.named_modules():
        if isinstance(mod, FORBIDDEN_TYPES):
            raise TypeError(f"Forbidden NPU op '{name}': {mod.__class__.__name__}")


def summarize_npu_ops(model: nn.Module) -> dict[str, int]:
    ops: dict[str, int] = defaultdict(int)
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            key = 'DWConv' if module.groups == module.in_channels and module.in_channels > 1 else 'Conv'
            ops[key] += 1
        elif isinstance(module, nn.BatchNorm2d):
            ops['BN'] += 1
        elif isinstance(module, nn.PReLU):
            ops['PReLU'] += 1
        elif isinstance(module, nn.AdaptiveAvgPool2d):
            ops['AdaptiveAvgPool2d'] += 1
        elif isinstance(module, nn.Hardsigmoid):
            ops['HardSigmoid'] += 1
        elif isinstance(module, nn.Dropout2d):
            ops['Dropout2d'] += 1
    return dict(ops)


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)


def charbonnier_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    diff = pred - target
    return torch.mean(torch.sqrt(diff * diff + eps * eps))


def combined_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    return 0.5 * charbonnier_loss(pred, target, eps) + 0.5 * F.l1_loss(pred, target)


def compute_psnr(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    pred = pred.clamp(0.0, 1.0)
    target = target.clamp(0.0, 1.0)
    mse = torch.mean((pred - target) ** 2, dim=(1, 2, 3)).clamp_min(1e-12)
    return -10.0 * torch.log10(mse)


class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model: nn.Module) -> None:
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name].mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)

    def apply_shadow(self, model: nn.Module) -> None:
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self, model: nn.Module) -> None:
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name])
        self.backup = {}


def lr_for_epoch(epoch: int, total: int, base_lr: float, warmup: int, min_ratio: float) -> float:
    if warmup > 0 and epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(1, total - warmup - 1)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_lr = base_lr * min_ratio
    return min_lr + (base_lr - min_lr) * cosine
""")
exec(COMMON_SOURCE, globals())

MODEL_SOURCE = textwrap.dedent(r"""
class StochasticDepth(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x):
        if not self.training or self.drop_prob <= 0.0:
            return x
        keep = 1.0 - self.drop_prob
        mask = keep + torch.rand((x.shape[0],) + (1,) * (x.ndim - 1), dtype=x.dtype, device=x.device)
        mask.floor_()
        return x * mask / keep


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, mid, 1, bias=True)
        self.act = nn.PReLU(num_parameters=mid)
        self.fc2 = nn.Conv2d(mid, channels, 1, bias=True)
        self.gate = nn.Hardsigmoid()

    def forward(self, x):
        weights = self.gate(self.fc2(self.act(self.fc1(self.pool(x)))))
        return x * weights


class SEResBlock(nn.Module):
    def __init__(self, channels, reduction=4, dropout=0.0, drop_path=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.act = nn.PReLU(num_parameters=channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.se = SEBlock(channels, reduction)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0.0 else nn.Identity()
        self.drop_path = StochasticDepth(drop_path)

    def forward(self, x):
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out = self.drop_path(self.dropout(out))
        return x + out


class WideSEResNetSR(nn.Module):
    def __init__(self, num_blocks=16, channels=64, reduction=4, dropout=0.08, max_drop_path=0.10):
        super().__init__()
        drops = torch.linspace(0.0, max_drop_path, num_blocks).tolist()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.PReLU(num_parameters=channels),
        )
        self.body = nn.Sequential(*[
            SEResBlock(channels, reduction, dropout, drops[i]) for i in range(num_blocks)
        ])
        self.tail = nn.Conv2d(channels, 3, 3, padding=1)

    def forward(self, x):
        return x + self.tail(self.body(self.stem(x)))


MODEL_CFG = {
    'num_blocks': 16,
    'channels': 64,
    'reduction': 4,
    'dropout': 0.08,
    'max_drop_path': 0.10,
}


def build_model(**model_cfg):
    return WideSEResNetSR(**model_cfg)


prepare_export_model = None
""")
exec(MODEL_SOURCE, globals())

EXPORT_SOURCE = textwrap.dedent(r"""
def default_calibration_cfg(seed: int) -> dict[str, Any]:
    return {'num_samples': 128, 'seed': seed, 'output_subdir': 'calibration'}


def load_checkpoint(model: nn.Module, checkpoint_path: Path, map_location: str | torch.device = 'cpu'):
    ckpt = torch.load(checkpoint_path, map_location=map_location)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'])
    else:
        model.load_state_dict(ckpt)
    return ckpt


def load_history(metrics_path: Path) -> list[dict[str, Any]]:
    if not metrics_path.exists():
        return []
    return [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]


@torch.no_grad()
def collect_psnr_records(model: nn.Module, loader: DataLoader, device: torch.device, max_items: int | None = None):
    model.eval()
    records = []
    for lr_img, hr_img, names, sources in loader:
        lr_img = lr_img.to(device)
        hr_img = hr_img.to(device)
        pred = model(lr_img)
        pred_psnr = compute_psnr(pred, hr_img).cpu().tolist()
        input_psnr = compute_psnr(lr_img, hr_img).cpu().tolist()
        for name, source, pred_p, input_p in zip(names, sources, pred_psnr, input_psnr):
            records.append({'name': name, 'source': source, 'pred_psnr': float(pred_p), 'input_psnr': float(input_p)})
            if max_items is not None and len(records) >= max_items:
                return records
    return records


def summarize_records(title: str, records: list[dict[str, Any]]) -> None:
    if not records:
        print(f"{title}: no records")
        return
    psnrs = torch.tensor([row['pred_psnr'] for row in records], dtype=torch.float32)
    baselines = torch.tensor([row['input_psnr'] for row in records], dtype=torch.float32)
    print(f"{title}: n={len(records)} | model={psnrs.mean():.3f} dB | baseline={baselines.mean():.3f} dB")
    print(
        f"  p10={torch.quantile(psnrs, 0.1):.3f} | "
        f"p50={torch.quantile(psnrs, 0.5):.3f} | "
        f"p90={torch.quantile(psnrs, 0.9):.3f}"
    )
    for row in sorted(records, key=lambda item: item['pred_psnr'])[:3]:
        print(f"  hardest: {row['source']}:{row['name']} -> {row['pred_psnr']:.3f} dB")


def run_diagnostics(
    build_model: Any,
    model_cfg: dict[str, Any],
    output_dir: Path,
    data_bundle: dict[str, Any],
    device: torch.device,
    prepare_export_model: Any | None = None,
) -> None:
    best_ckpt = Path(output_dir) / 'best.pt'
    if not best_ckpt.exists():
        print(f"Run TPU training first so {best_ckpt} exists.")
        return
    diag_model = build_model(**model_cfg).to(device)
    ckpt = load_checkpoint(diag_model, best_ckpt, map_location=device)
    if 'ema_shadow' in ckpt:
        for name, param in diag_model.named_parameters():
            if name in ckpt['ema_shadow']:
                param.data.copy_(ckpt['ema_shadow'][name])
    if prepare_export_model is not None:
        diag_model = prepare_export_model(diag_model)
    summarize_records('train_eval', collect_psnr_records(diag_model, data_bundle['train_eval_loader'], device, max_items=128))
    summarize_records('paired_val', collect_psnr_records(diag_model, data_bundle['paired_val_loader'], device))
    summarize_records('imagenet_val', collect_psnr_records(diag_model, data_bundle['imagenet_val_loader'], device))
    summarize_records('combined_val', collect_psnr_records(diag_model, data_bundle['val_loader'], device))


def export_to_onnx(
    build_model: Any,
    model_cfg: dict[str, Any],
    checkpoint_path: Path,
    onnx_path: Path,
    data_cfg: dict[str, Any],
    device: torch.device,
    prepare_export_model: Any | None = None,
    verify: bool = False,
    sample_loader: DataLoader | None = None,
) -> Path:
    checkpoint_path = Path(checkpoint_path)
    onnx_path = Path(onnx_path)
    onnx_path.parent.mkdir(parents=True, exist_ok=True)

    export_model = build_model(**model_cfg).to(device)
    ckpt = load_checkpoint(export_model, checkpoint_path, map_location=device)
    if 'ema_shadow' in ckpt:
        for name, param in export_model.named_parameters():
            if name in ckpt['ema_shadow']:
                param.data.copy_(ckpt['ema_shadow'][name])
        print('Loaded EMA weights for export')
    if prepare_export_model is not None:
        export_model = prepare_export_model(export_model)
    export_model.eval()

    if 'metrics' in ckpt:
        val_psnr = ckpt['metrics'].get('val_psnr', None)
        if val_psnr is None:
            print(f"Checkpoint epoch {ckpt['metrics'].get('epoch', '?')}")
        else:
            print(f"Checkpoint epoch {ckpt['metrics'].get('epoch', '?')}, val_psnr={val_psnr:.3f} dB")

    dummy = torch.randn(1, 3, data_cfg['eval_size'], data_cfg['eval_size'], device=device)
    export_kwargs = dict(
        export_params=True,
        opset_version=13,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
    )
    try:
        torch.onnx.export(export_model, dummy, str(onnx_path), dynamo=False, **export_kwargs)
    except TypeError:
        torch.onnx.export(export_model, dummy, str(onnx_path), **export_kwargs)

    print(f"Exported to {onnx_path} ({onnx_path.stat().st_size / 1024:.0f} KB)")
    if onnx is not None:
        onnx.checker.check_model(onnx.load(str(onnx_path)))
        print('ONNX checker: passed')

    if verify and ort is not None and sample_loader is not None:
        sample_lr, _, _, _ = next(iter(sample_loader))
        sample_lr = sample_lr[:1].to(device)
        with torch.no_grad():
            torch_out = export_model(sample_lr).cpu().numpy()
        sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
        ort_out = sess.run(['output'], {'input': sample_lr.cpu().numpy()})[0]
        diff = abs(torch_out - ort_out)
        print(f"Parity: max_diff={diff.max():.8f}, mean_diff={diff.mean():.8f}")
    return onnx_path


def slugify_name(text: str) -> str:
    cleaned = ''.join(ch if ch.isalnum() else '_' for ch in text)
    return '_'.join(part for part in cleaned.split('_') if part)[:80] or 'sample'


@torch.no_grad()
def score_lr_tensor(lr_t: torch.Tensor) -> dict[str, float]:
    gray = lr_t.float().mean(dim=0)
    brightness = float(gray.mean().item())
    contrast = float(gray.std().item())
    grad_x = gray[:, 1:] - gray[:, :-1]
    grad_y = gray[1:, :] - gray[:-1, :]
    texture = float(0.5 * (grad_x.abs().mean().item() + grad_y.abs().mean().item()))
    return {'brightness': brightness, 'contrast': contrast, 'texture': texture}


@torch.no_grad()
def collect_calibration_candidates(dataset_map: dict[str, Dataset]) -> list[dict[str, Any]]:
    records = []
    for dataset_key, dataset in dataset_map.items():
        for dataset_index in range(len(dataset)):
            lr_t, _, name, source = dataset[dataset_index]
            records.append(
                {
                    'dataset_key': dataset_key,
                    'dataset_index': dataset_index,
                    'name': name,
                    'source': source,
                    **score_lr_tensor(lr_t),
                }
            )
    return records


def assign_tertile_bins(records: list[dict[str, Any]], metric: str) -> None:
    vals = torch.tensor([row[metric] for row in records], dtype=torch.float32)
    lo, hi = torch.quantile(vals, 1 / 3).item(), torch.quantile(vals, 2 / 3).item()
    for row in records:
        row[f'{metric}_bin'] = 'low' if row[metric] <= lo else ('high' if row[metric] >= hi else 'mid')


def select_diverse_calibration_subset(records: list[dict[str, Any]], num_samples: int, seed: int) -> list[dict[str, Any]]:
    if not records:
        return []
    tagged = [dict(row) for row in records]
    assign_tertile_bins(tagged, 'brightness')
    assign_tertile_bins(tagged, 'texture')
    rng = random.Random(seed)
    buckets: dict[tuple[str, str, str], list[dict[str, Any]]] = defaultdict(list)
    for row in tagged:
        buckets[(row['source'], row['brightness_bin'], row['texture_bin'])].append(row)
    keys = sorted(buckets)
    for key in keys:
        rng.shuffle(buckets[key])

    target = min(num_samples, len(tagged))
    selected, seen = [], set()
    while len(selected) < target:
        progress = False
        for key in keys:
            while buckets[key]:
                row = buckets[key].pop()
                row_id = (row['dataset_key'], row['dataset_index'])
                if row_id in seen:
                    continue
                seen.add(row_id)
                selected.append(row)
                progress = True
                break
            if len(selected) >= target:
                break
        if not progress:
            break
    return selected


def export_calibration_artifacts(records: list[dict[str, Any]], dataset_map: dict[str, Dataset], output_dir: Path, cfg: dict[str, Any]) -> Path:
    cal_dir = Path(output_dir) / cfg['output_subdir']
    cal_dir.mkdir(parents=True, exist_ok=True)
    manifest = []
    batch = []
    for row in records:
        lr_t, _, name, source = dataset_map[row['dataset_key']][row['dataset_index']]
        batch.append(lr_t)
        img_path = cal_dir / f"{len(manifest):03d}_{slugify_name(source)}_{slugify_name(name)}.png"
        transforms.ToPILImage()(lr_t).save(img_path)
        manifest.append(
            {
                'dataset_key': row['dataset_key'],
                'dataset_index': row['dataset_index'],
                'name': name,
                'source': source,
                'image_path': str(img_path),
                'brightness': row['brightness'],
                'contrast': row['contrast'],
                'texture': row['texture'],
            }
        )
    tensor_path = cal_dir / 'calibration_inputs.pt'
    torch.save(torch.stack(batch), tensor_path)
    summary = {
        'num_samples': len(manifest),
        'brightness_mean': sum(row['brightness'] for row in records) / max(1, len(records)),
        'texture_mean': sum(row['texture'] for row in records) / max(1, len(records)),
    }
    (cal_dir / 'manifest.json').write_text(json.dumps({'config': cfg, 'summary': summary, 'samples': manifest}, indent=2))
    print(f"Calibration: {len(records)} samples exported to {cal_dir}")
    print(f"Calibration manifest: {cal_dir / 'manifest.json'}")
    print(f"Calibration tensor batch: {tensor_path}")
    return cal_dir


def export_default_calibration(data_bundle: dict[str, Any], output_dir: Path, cfg: dict[str, Any]) -> Path:
    candidates = collect_calibration_candidates(data_bundle['calibration_datasets'])
    selected = select_diverse_calibration_subset(candidates, cfg['num_samples'], cfg['seed'])
    return export_calibration_artifacts(selected, data_bundle['calibration_datasets'], output_dir, cfg)
""")
exec(EXPORT_SOURCE, globals())

TPU_TRAINING_SOURCE = textwrap.dedent(r"""
def xla_world_size() -> int:
    try:
        return int(xr.world_size())
    except Exception:
        return 1


def xla_rank() -> int:
    try:
        return int(xr.global_ordinal())
    except Exception:
        return 0


def is_master() -> bool:
    return xla_rank() == 0


def to_cpu(obj):
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {key: to_cpu(value) for key, value in obj.items()}
    if isinstance(obj, list):
        return [to_cpu(value) for value in obj]
    if isinstance(obj, tuple):
        return tuple(to_cpu(value) for value in obj)
    return obj


def default_prefetch_factor() -> int:
    return 2


def per_process_num_workers(host_workers: int, world_size: int) -> int:
    return max(1, host_workers // max(1, world_size))


def make_optimizer(model: nn.Module, train_cfg: dict[str, Any]):
    optimizer_cls = AdamW
    if 'syncfree_optim' in globals() and syncfree_optim is not None and hasattr(syncfree_optim, 'AdamW'):
        optimizer_cls = syncfree_optim.AdamW
    return optimizer_cls(model.parameters(), lr=train_cfg['lr'], weight_decay=train_cfg['weight_decay'])


def reduced_means(loss_sum: torch.Tensor, psnr_sum: torch.Tensor, count: torch.Tensor) -> tuple[float, float]:
    if xla_world_size() > 1:
        xm.all_reduce(xm.REDUCE_SUM, [loss_sum, psnr_sum, count])
    count = count.clamp_min(1.0)
    return float((loss_sum / count).item()), float((psnr_sum / count).item())


def train_one_epoch_xla(
    model: nn.Module,
    loader,
    optimizer,
    train_cfg: dict[str, Any],
    ema: EMA | None,
    device: torch.device,
    epoch_num: int,
) -> dict[str, float]:
    model.train()
    loss_sum = torch.zeros((), device=device)
    psnr_sum = torch.zeros((), device=device)
    count = torch.zeros((), device=device)
    for lr_img, hr_img, _, _ in loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('xla', dtype=torch.bfloat16, enabled=train_cfg['use_amp']):
            pred = model(lr_img)
            loss = combined_loss(pred, hr_img, eps=train_cfg['charb_eps'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=train_cfg['grad_clip_norm'])
        xm.optimizer_step(optimizer)
        if ema is not None:
            ema.update(model)
        batch_size = float(lr_img.shape[0])
        loss_sum += loss.detach() * batch_size
        psnr_sum += compute_psnr(pred.detach(), hr_img).sum().detach()
        count += torch.tensor(batch_size, device=device)
    train_loss, train_psnr = reduced_means(loss_sum, psnr_sum, count)
    return {'train_loss': train_loss, 'train_psnr_online': train_psnr}


@torch.no_grad()
def evaluate_loader_xla(
    model: nn.Module,
    loader,
    train_cfg: dict[str, Any],
    device: torch.device,
    split_name: str,
) -> dict[str, float]:
    model.eval()
    loss_sum = torch.zeros((), device=device)
    psnr_sum = torch.zeros((), device=device)
    count = torch.zeros((), device=device)
    for lr_img, hr_img, _, _ in loader:
        with torch.autocast('xla', dtype=torch.bfloat16, enabled=train_cfg['use_amp']):
            pred = model(lr_img)
            loss = combined_loss(pred, hr_img, eps=train_cfg['charb_eps'])
        batch_size = float(lr_img.shape[0])
        loss_sum += loss.detach() * batch_size
        psnr_sum += compute_psnr(pred, hr_img).sum().detach()
        count += torch.tensor(batch_size, device=device)
    avg_loss, avg_psnr = reduced_means(loss_sum, psnr_sum, count)
    return {f'{split_name}_loss': avg_loss, f'{split_name}_psnr': avg_psnr}


def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer,
    epoch: int,
    metrics: dict[str, Any],
    best_psnr: float,
    best_epoch: int,
    model_cfg: dict[str, Any],
    train_cfg: dict[str, Any],
    data_cfg: dict[str, Any],
    paths_cfg: dict[str, Any],
    ema: EMA | None = None,
) -> None:
    state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
        'best_psnr': best_psnr,
        'best_epoch': best_epoch,
        'model_cfg': model_cfg,
        'train_cfg': train_cfg,
        'data_cfg': data_cfg,
        'paths_cfg': paths_cfg,
    }
    if ema is not None:
        state['ema_shadow'] = ema.shadow
    if is_master():
        torch.save(to_cpu(state), path)


def should_run_train_eval(epoch_num: int, train_cfg: dict[str, Any]) -> bool:
    interval = train_cfg.get('train_eval_interval', 5)
    if epoch_num == train_cfg['epochs']:
        return True
    if interval > 0 and epoch_num % interval == 0:
        return True
    checkpoint_interval = train_cfg.get('checkpoint_interval', 10)
    return checkpoint_interval > 0 and epoch_num % checkpoint_interval == 0


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text())


def write_jsonl(path: Path, row: dict[str, Any]) -> None:
    with path.open('a') as handle:
        handle.write(json.dumps(row) + '\n')


def fit_tpu(config: dict[str, Any]) -> None:
    output_dir = Path(config['workspace']['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / 'metrics.jsonl'
    last_ckpt_path = output_dir / 'last.pt'
    best_ckpt_path = output_dir / 'best.pt'

    data_cfg = config['data_cfg']
    train_cfg = config['train_cfg']
    paths_cfg = config['paths_cfg']
    model_cfg = config['model_cfg']
    host_workers = int(config['host_dataloader_workers'])

    device = xm.xla_device()
    rank = xla_rank()
    world_size = xla_world_size()
    if train_cfg['global_batch_size'] % max(1, world_size) != 0:
        raise ValueError(
            f"GLOBAL_BATCH_SIZE={train_cfg['global_batch_size']} must be divisible by world size {world_size}"
        )
    per_device_batch = train_cfg['global_batch_size'] // max(1, world_size)
    workers = per_process_num_workers(host_workers, world_size)
    set_seed(train_cfg['seed'])

    bundle = build_datasets(paths_cfg, data_cfg, seed=train_cfg['seed'])
    subset_size = min(data_cfg['train_eval_subset_size'], len(bundle['train_eval_dataset']))
    subset_indices = list(range(subset_size))
    train_eval_subset = Subset(bundle['train_eval_dataset'], subset_indices)

    train_sampler = DistributedSampler(
        bundle['train_dataset'],
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        drop_last=True,
    )
    train_eval_sampler = DistributedSampler(
        train_eval_subset,
        num_replicas=world_size,
        rank=rank,
        shuffle=False,
        drop_last=False,
    )
    val_sampler = DistributedSampler(
        bundle['val_dataset'],
        num_replicas=world_size,
        rank=rank,
        shuffle=False,
        drop_last=False,
    )

    loader_kw = loader_kwargs(workers, pin_memory=False, prefetch_factor=default_prefetch_factor())
    train_loader = DataLoader(bundle['train_dataset'], batch_size=per_device_batch, sampler=train_sampler, drop_last=True, **loader_kw)
    train_eval_loader = DataLoader(train_eval_subset, batch_size=per_device_batch, sampler=train_eval_sampler, drop_last=False, **loader_kw)
    val_loader = DataLoader(bundle['val_dataset'], batch_size=per_device_batch, sampler=val_sampler, drop_last=False, **loader_kw)

    model = build_model(**model_cfg).to(device)
    assert_npu_compatible(model)
    optimizer = make_optimizer(model, train_cfg)
    ema = EMA(model, decay=train_cfg['ema_decay'])

    start_epoch = 0
    best_psnr = float('-inf')
    best_epoch = -1
    epochs_without_improve = 0

    if config.get('resume_training', True) and last_ckpt_path.exists():
        ckpt = torch.load(last_ckpt_path, map_location='cpu')
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = int(ckpt['epoch'])
        best_psnr = float(ckpt.get('best_psnr', float('-inf')))
        best_epoch = int(ckpt.get('best_epoch', -1))
        if 'ema_shadow' in ckpt:
            ema.shadow = {name: tensor.to(device) for name, tensor in ckpt['ema_shadow'].items()}
        if is_master():
            print(f"RESUMING from epoch {start_epoch}/{train_cfg['epochs']}")
            print(f"  Best PSNR so far: {best_psnr:.3f} dB (epoch {best_epoch})")
    else:
        if is_master():
            metrics_path.write_text('')
            print('FRESH START')

    if start_epoch >= train_cfg['epochs']:
        if is_master():
            print(f"Already trained {start_epoch} epochs. Increase epochs to continue.")
        return

    if is_master():
        print(f"Using TPU world size: {world_size}")
        print(f"Per-device batch size: {per_device_batch}")
        print(f"Per-process dataloader workers: {workers}")
        print(f"Training model parameters: {count_parameters(model):,}")
        print(f"Training model ops: {summarize_npu_ops(model)}")
        print(f"\n{'epoch':>5} | {'lr':>8} | {'train_loss':>10} {'train_online':>12} {'train_eval':>11} | {'val_psnr':>8} | {'best':>8} | {'time':>7}")
        print('-' * 98)

    train_device_loader = pl.MpDeviceLoader(train_loader, device)
    train_eval_device_loader = pl.MpDeviceLoader(train_eval_loader, device)
    val_device_loader = pl.MpDeviceLoader(val_loader, device)

    for epoch in range(start_epoch, train_cfg['epochs']):
        epoch_num = epoch + 1
        epoch_lr = lr_for_epoch(epoch, train_cfg['epochs'], train_cfg['lr'], train_cfg['warmup_epochs'], train_cfg['min_lr_ratio'])
        for group in optimizer.param_groups:
            group['lr'] = epoch_lr
        train_sampler.set_epoch(epoch_num)

        start_time = time.time()
        train_metrics = train_one_epoch_xla(model, train_device_loader, optimizer, train_cfg, ema, device, epoch_num)
        ema.apply_shadow(model)
        row = {'epoch': epoch_num, 'lr': epoch_lr}
        row.update(train_metrics)
        if should_run_train_eval(epoch_num, train_cfg):
            row.update(evaluate_loader_xla(model, train_eval_device_loader, train_cfg, device, 'train_eval'))
        row.update(evaluate_loader_xla(model, val_device_loader, train_cfg, device, 'val'))
        ema.restore(model)
        seconds = round(time.time() - start_time, 1)
        row['seconds'] = seconds

        val_psnr = row['val_psnr']
        improved = val_psnr > best_psnr
        if improved:
            best_psnr = val_psnr
            best_epoch = epoch_num
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1

        save_checkpoint(last_ckpt_path, model, optimizer, epoch_num, row, best_psnr, best_epoch, model_cfg, train_cfg, data_cfg, paths_cfg, ema=ema)
        if improved:
            save_checkpoint(best_ckpt_path, model, optimizer, epoch_num, row, best_psnr, best_epoch, model_cfg, train_cfg, data_cfg, paths_cfg, ema=ema)
        if epoch_num % train_cfg['checkpoint_interval'] == 0 or epochs_without_improve >= train_cfg['early_stop_patience']:
            save_checkpoint(output_dir / f'epoch_{epoch_num:03d}.pt', model, optimizer, epoch_num, row, best_psnr, best_epoch, model_cfg, train_cfg, data_cfg, paths_cfg, ema=ema)

        xm.rendezvous(f'checkpoint_epoch_{epoch_num}')
        if is_master():
            write_jsonl(metrics_path, row)
            train_eval_str = f"{row['train_eval_psnr']:.3f}" if 'train_eval_psnr' in row else '     -'
            mark = '*' if improved else ' '
            print(
                f"{epoch_num:5d} | {epoch_lr:8.2e} | {row['train_loss']:10.6f} {row['train_psnr_online']:12.3f} {train_eval_str:>11} | "
                f"{row['val_psnr']:8.3f} | {best_psnr:8.3f} | {seconds:6.1f}s {mark}"
            )
        if epochs_without_improve >= train_cfg['early_stop_patience']:
            if is_master():
                print(f"Early stopping after {train_cfg['early_stop_patience']} epochs without improvement.")
            break

    if is_master():
        print(f"\nTraining complete. Best epoch {best_epoch} with val_psnr={best_psnr:.3f} dB")
        print(f"Checkpoints: {output_dir}")


def worker_main(index: int, config_path: str) -> None:
    config = load_json(Path(config_path))
    fit_tpu(config)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    args = parser.parse_args()
    config = load_json(Path(args.config))
    if config.get('xla_debug_single_process', False):
        worker_main(0, args.config)
        return
    launch = getattr(torch_xla, 'launch', None)
    if launch is not None:
        launch(worker_main, args=(args.config,))
        return
    import torch_xla.distributed.xla_multiprocessing as xmp

    xmp.spawn(worker_main, args=(args.config,), nprocs=8, start_method='fork')


if __name__ == '__main__':
    main()
""")


def render_tpu_training_script() -> str:
    header = textwrap.dedent(r"""
from __future__ import annotations

import argparse
import hashlib
import io
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFilter, ImageOps
from torch.optim import AdamW
from torch.utils.data import ConcatDataset, DataLoader, Dataset, DistributedSampler, Subset
from torchvision import transforms

import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
import torch_xla.distributed.parallel_loader as pl

try:
    import torch_xla.amp.syncfree as syncfree_optim
except Exception:
    syncfree_optim = None
""")
    return "\n\n".join([header.strip(), COMMON_SOURCE.strip(), MODEL_SOURCE.strip(), TPU_TRAINING_SOURCE.strip()]) + "\n"


## 5. Model Definition


In [ ]:

# Model definition comes from MODEL_SOURCE above.
print(f"Model config: {MODEL_CFG}")


In [ ]:

summary_device = torch.device("cpu")
summary_data_cfg = default_data_cfg()

model = build_model(**MODEL_CFG).to(summary_device)
assert_npu_compatible(model)
with torch.no_grad():
    dummy = torch.randn(1, 3, summary_data_cfg["train_patch_size"], summary_data_cfg["train_patch_size"], device=summary_device)
    out = model(dummy)

print(f"Model: WideSEResNetSR | Config: {MODEL_CFG}")
print(f"Parameters: {count_parameters(model):,}")
print(f"Input: {tuple(dummy.shape)} -> Output: {tuple(out.shape)}")
print(f"NPU ops: {summarize_npu_ops(model)}")


## 6. Training Config


In [ ]:

cpu_device = torch.device("cpu")

data_cfg = default_data_cfg()
data_cfg["imagenet_train_limit"] = IMAGENET_TRAIN_LIMIT
data_cfg["imagenet_val_limit"] = IMAGENET_VAL_LIMIT
data_cfg["train_eval_subset_size"] = TRAIN_EVAL_SUBSET_SIZE

train_cfg = default_train_cfg(GLOBAL_BATCH_SIZE)
train_cfg["epochs"] = EPOCHS
train_cfg["checkpoint_interval"] = CHECKPOINT_INTERVAL
train_cfg["train_eval_interval"] = TRAIN_EVAL_INTERVAL
train_cfg["early_stop_patience"] = EARLY_STOP_PATIENCE
train_cfg["seed"] = SEED
train_cfg["use_amp"] = True

calibration_cfg = default_calibration_cfg(SEED)
export_batch_size = max(1, min(CPU_EXPORT_BATCH_SIZE, GLOBAL_BATCH_SIZE))
export_data_bundle = build_data_bundle(
    paths_cfg=paths_cfg,
    data_cfg=data_cfg,
    batch_size=export_batch_size,
    num_workers=0,
    seed=SEED,
    pin_memory=False,
)
print_data_summary(export_data_bundle)

training_config = {
    "workspace": {
        "workspace_root": str(workspace["workspace_root"]),
        "output_dir": str(workspace["output_dir"]),
    },
    "paths_cfg": paths_cfg,
    "data_cfg": data_cfg,
    "train_cfg": train_cfg,
    "model_cfg": MODEL_CFG,
    "resume_training": RESUME_TRAINING,
    "host_dataloader_workers": HOST_DATALOADER_WORKERS,
    "xla_debug_single_process": XLA_DEBUG_SINGLE_PROCESS,
}

print(json.dumps({
    "train_cfg": train_cfg,
    "data_cfg": data_cfg,
    "calibration_cfg": calibration_cfg,
    "global_batch_size": GLOBAL_BATCH_SIZE,
    "host_dataloader_workers": HOST_DATALOADER_WORKERS,
    "cpu_export_batch_size": export_batch_size,
    "xla_debug_single_process": XLA_DEBUG_SINGLE_PROCESS,
}, indent=2))


## 7. Full Training Loop


In [ ]:

probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import json; import torch_xla.core.xla_model as xm; print(json.dumps({'supported_devices': xm.get_xla_supported_devices()}, indent=2))",
    ],
    capture_output=True,
    text=True,
)
if probe.stdout.strip():
    print(probe.stdout)
if probe.returncode != 0:
    print(probe.stderr)
    raise subprocess.CalledProcessError(probe.returncode, probe.args)

script_path = Path(workspace["workspace_root"]) / "phase5a_wide_se_tpu_train.py"
config_path = Path(workspace["workspace_root"]) / "phase5a_wide_se_tpu_train_config.json"
script_path.write_text(render_tpu_training_script())
config_path.write_text(json.dumps(training_config, indent=2))
print(f"Wrote TPU training script: {script_path}")
print(f"Wrote TPU config: {config_path}")

subprocess.check_call([sys.executable, str(script_path), "--config", str(config_path)])

metrics_path = Path(workspace["output_dir"]) / "metrics.jsonl"
if metrics_path.exists():
    rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    if rows:
        best_row = max(rows, key=lambda row: row["val_psnr"])
        print(json.dumps({
            "best_epoch": best_row["epoch"],
            "best_val_psnr": round(best_row["val_psnr"], 4),
            "last_epoch": rows[-1]["epoch"],
            "last_val_psnr": round(rows[-1]["val_psnr"], 4),
        }, indent=2))


## 8. Diagnostics And PSNR Summary


In [ ]:

metrics_path = Path(workspace["output_dir"]) / "metrics.jsonl"
if metrics_path.exists():
    rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    if rows:
        best_row = max(rows, key=lambda row: row["val_psnr"])
        print(json.dumps({
            "best_epoch": best_row["epoch"],
            "best_val_psnr": round(best_row["val_psnr"], 4),
            "last_epoch": rows[-1]["epoch"],
            "last_val_psnr": round(rows[-1]["val_psnr"], 4),
        }, indent=2))

run_diagnostics(
    build_model=build_model,
    model_cfg=MODEL_CFG,
    output_dir=Path(workspace["output_dir"]),
    data_bundle=export_data_bundle,
    device=cpu_device,
    prepare_export_model=prepare_export_model,
)


## 9. ONNX Export


In [ ]:

best_ckpt = Path(workspace["output_dir"]) / "best.pt"
onnx_path = export_to_onnx(
    build_model=build_model,
    model_cfg=MODEL_CFG,
    checkpoint_path=best_ckpt,
    onnx_path=Path(workspace["output_dir"]) / "best.onnx",
    data_cfg=data_cfg,
    device=cpu_device,
    prepare_export_model=prepare_export_model,
    verify=True,
    sample_loader=export_data_bundle["paired_val_loader"],
)
print(f"Ready for separate MXQ compilation: {onnx_path}")


## 10. Calibration Export

MXQ compilation stays outside this notebook and should be run separately with `onnx_to_mxq.py`.


In [ ]:

calibration_dir = export_default_calibration(export_data_bundle, Path(workspace["output_dir"]), calibration_cfg)
print(f"Calibration ready: {calibration_dir}")
print("MXQ compilation is a separate post-export step via onnx_to_mxq.py.")
